In [1]:
# import galstreams
# mws = galstreams.MWStreams(verbose=False)

import sys, pickle, os
from importlib import reload
from tqdm import tqdm, trange

import numpy as np, pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm, LogNorm

import agama 
Gyr_to_AgamaTime = 1.0227 # 1 Gyr in Agama time units (kpc/(km/s))
import nbody_streams.agama_helper as ah

import emcee
import corner

from astropy import units as u
import astropy.coordinates as coord
from astropy.coordinates import Galactocentric, ICRS, CartesianDifferential, CartesianRepresentation
from astropy import table

In [2]:
os.getcwd()

'/astro/users/shriyp/mw_gpotential_streams/notebooks'

In [3]:
sys.path.append('../scripts/')
from coordinate_utils import get_rotation_matrix, sf_to_icrs, icrs_to_sf
from generate_sim_stream import create_stream_particle_spray
from stream_likelihood import log_likelihood, make_spline, log_prior, log_probability
from stream_data_utils import read_in_data

In [4]:
aau_member_path = '../data/aau_members.csv'
aau_distance_path = '../data/aau_bhb_rrl.csv'

aau_table = pd.read_csv(aau_member_path)
aau_bhb_rrl_data = pd.read_csv(aau_distance_path)

aau_bhb_rrl_data.columns = aau_bhb_rrl_data.columns.str.lower()

df_aau, prog_pars_aau, prog_pars_icrs_aau, df_distance_aau = read_in_data(aau_table, 'distance_modulus', aau_bhb_rrl_data)

prog_pars_aau[2] = 10**(((prog_pars_aau[2]) + 5) / 5) / 1000.0
prog_pars_icrs_aau[2] = 10**(((prog_pars_icrs_aau[2]) + 5) / 5) / 1000.0 

jet_member_path = '../data/jet_members.csv'

jet_table = pd.read_csv(jet_member_path)

df_jet, distance_fit_jet, prog_pars_jet, prog_pars_icrs_jet = read_in_data(jet_table, 'bhb_dist')

In [5]:
data_dict_jet = dict(
    phi1_obs = df_jet['phi1'].values,
    phi2_obs = df_jet['phi2'].values,
    rv_obs = df_jet['vel_calib'].values,
    rv_obs_errors = df_jet['vel_calib_std'].values,
    dist_obs = df_jet['bhb_dist'].values,
    dist_obs_errors = (df_jet['bhb_dist'].values*0.1),
    pmra_cosdec_obs = df_jet['pmra'].values,
    pmra_cosdec_obs_errors = df_jet['pmra_error'].values,
    pmdec_obs = df_jet['pmdec'].values,
    pmdec_obs_errors = df_jet['pmdec_error'].values,
)

In [6]:
BASE_POT_PATH = '../potential_files/'
potMW_path = os.path.join(BASE_POT_PATH, 'base_mw_pot.ini')
## potential models to load
basePot = agama.Potential(file=potMW_path)
scaleNFW_half = agama.Potential(type = 'NFW', mass = 5.5427e11, scaleRadius = 15.626, scale= [[-5, 0.5, 1],[0, 1, 1]])
potHalf= basePot + scaleNFW_half
NFW = agama.Potential(type= 'NFW', mass = 5.5427e11, scaleRadius = 15.626)
potStat = basePot + NFW
#pot_spheroid.density([10, 0, 0], t=0), pot_spheroid.density([10, 0, 0], t=-5)
#pot_spheroid = agama.Potential(type='spheroid', mass=1e12, scaleradius=20, scale=[[-5, 0.5, 1],[0, 1, 1]])

In [7]:
print(potHalf)
print(potStat)

CompositePotential{ MiyamotoNagai, MiyamotoNagai, MiyamotoNagai, Dehnen, Dehnen, Scaled NFW } (symmetry: Axisymmetric)
CompositePotential{ MiyamotoNagai, MiyamotoNagai, MiyamotoNagai, Dehnen, Dehnen, NFW } (symmetry: Axisymmetric)


In [8]:
#Constructing stream progenitor present-day coordinates and information

prog_mass, prog_scaleradius =  20_000, 10/1_000 # Msun, kpc

num_particles = 3_000

Age_stream_inGyr = 5.0

R = np.array([[-0.69533693,  0.61240175, -0.37612584],
                    [-0.62909984, -0.26561497,  0.73053548],
                 [ 0.34747655,  0.744589,    0.56995374]])

jet_rot_matrix = u.Quantity(R, unit=u.dimensionless_unscaled)

#jet_rot_matrix = get_rotation_matrix('Jet-F22', mws = mws)
#print(jet_rot_matrix)

static_pot_bestfit_6D = [0, 0.15, 30.08, -0.64, -1.46, 272.41]

icrs_coord_ra, icrs_coord_dec = sf_to_icrs(static_pot_bestfit_6D[0], static_pot_bestfit_6D[1], jet_rot_matrix)
print(icrs_coord_ra, type(icrs_coord_ra))
print(icrs_coord_dec, type(icrs_coord_dec))

dist, pmra, pmdec, rv = static_pot_bestfit_6D[-4:]

jet_c = coord.SkyCoord(
        ra=icrs_coord_ra*u.degree, dec=icrs_coord_dec*u.degree, distance=dist*u.kpc, 
        pm_ra_cosdec=pmra*u.mas/u.yr,
        pm_dec=pmdec*u.mas/u.yr,
        radial_velocity=rv*u.km/u.s
    )
    
rep = jet_c.transform_to(coord.Galactocentric) # units here are kpc, km/s
    
prog_wtoday_gal = np.array(
        [rep.x.value, rep.y.value, rep.z.value,
         rep.v_x.value, rep.v_y.value, rep.v_z.value])

print(prog_wtoday_gal)

138.50121429303636 <class 'numpy.ndarray'>
-22.00159581100559 <class 'numpy.ndarray'>
[ -17.64031256  -26.95909359    9.37039313   -1.49156767  -91.40702591
 -102.2028583 ]


In [9]:
stream_model_half = create_stream_particle_spray(
    pot_host=potHalf, 
    initmass=prog_mass, 
    scaleradius=prog_scaleradius, 
    prog_pot_kind='Plummer', 
    sat_cen_present=prog_wtoday_gal, 
    num_particles=num_particles,
    time_end=0.0, 
    time_total=Age_stream_inGyr, save_rate=300,
    #add_perturber={'mass':0},
)
stream_model_stat = create_stream_particle_spray(
    pot_host=potStat, 
    initmass=prog_mass, 
    scaleradius=prog_scaleradius, 
    prog_pot_kind='Plummer', 
    sat_cen_present=prog_wtoday_gal, 
    num_particles=num_particles,
    time_end=0.0, 
    time_total=Age_stream_inGyr, save_rate=300,
    #add_perturber={'mass':0},
)

In [11]:
# #Making a movie
# n_particles_h, n_timesteps_h, __h = stream_model_half['part_xv'].shape

# n_particles_s, n_timesteps_s, __s = stream_model_stat['part_xv'].shape

# frames_dir = 'mg_half_mass_evolve_gif'
# os.makedirs(frames_dir, exist_ok=True)

# for t in range(n_timesteps_h):
#     #print(stream_model_half['part_xv'][:, 0, 0])  # x at first timestep
#     inc = (t*Age_stream_inGyr)/n_timesteps_h
#     age = inc-Age_stream_inGyr
#     #print(f"{age:.2f}")
    
#     fig = plt.figure(figsize=(8,8))
#     ax = fig.add_subplot(111, projection='3d')
#     ax.set_facecolor('#F6F6F6')
#     ax.xaxis.pane.fill = False
#     ax.yaxis.pane.fill = False
#     ax.zaxis.pane.fill = False
#     ax.xaxis.pane.set_edgecolor('none')
#     ax.yaxis.pane.set_edgecolor('none')
#     ax.zaxis.pane.set_edgecolor('none')
#     ax.grid(False)
#     ax.set_axis_off()

    
#     # plot all particles at this timestep
#     ax.scatter3D(
#         stream_model_half['part_xv'][:, t, 0],
#         stream_model_half['part_xv'][:, t, 1],
#         stream_model_half['part_xv'][:, t, 2],
#         s=2, c='tab:pink', alpha=0.6,label='Evolving'
#     )
#     ax.scatter3D(
#         stream_model_stat['part_xv'][:, t, 0],
#         stream_model_stat['part_xv'][:, t, 1],
#         stream_model_stat['part_xv'][:, t, 2],
#         s=2, c='#703BE7', alpha=0.6,label='Static'
#     )

#     # ax.scatter3D(
#     # prog_wtoday_gal[0], prog_wtoday_gal[1], prog_wtoday_gal[2],
#     # s=20, c='white', zorder=5
#     # )

#     # half-mass progenitor trail + dot
    
#     # half-mass progenitor trail
#     ax.plot3D(
#         stream_model_half['prog_xv'][:t+1, 0],
#         stream_model_half['prog_xv'][:t+1, 1],
#         stream_model_half['prog_xv'][:t+1, 2],
#         c='tab:pink', linestyle='--', linewidth=2, alpha=0.5
#     )
    
#     # static progenitor trail
#     ax.plot3D(
#         stream_model_stat['prog_xv'][:t+1, 0],
#         stream_model_stat['prog_xv'][:t+1, 1],
#         stream_model_stat['prog_xv'][:t+1, 2],
#         c='#703BE7', linestyle='--', linewidth=2, alpha=0.5
#     )
    
#     # two white dots, only label one to avoid duplicate legend entry
#     ax.scatter3D(
#         stream_model_half['prog_xv'][t, 0],
#         stream_model_half['prog_xv'][t, 1],
#         stream_model_half['prog_xv'][t, 2],
#         s=20, c='tab:pink', zorder=5
#     )
#     ax.scatter3D(
#         stream_model_stat['prog_xv'][t, 0],
#         stream_model_stat['prog_xv'][t, 1],
#         stream_model_stat['prog_xv'][t, 2],
#         s=20, c='#703BE7', zorder=5
#     )

#     legend = ax.legend(loc='upper left', fontsize=10, frameon=False)
#     for text in legend.get_texts():
#         text.set_color('black')

#     # white time label
#     ax.text2D(0.05, 0.05, f't = {age:.2f} Gyr', color='white', fontsize=12, transform=ax.transAxes)
    
#     # rotate angle with each frame
#     elev = 20 + 10 * np.sin(2 * np.pi * t / n_timesteps_h)
#     azim = (t / n_timesteps_h) * 360
#     ax.view_init(elev=elev, azim=azim)
    
#     fname = f'{frames_dir}/step_{t:04d}.png'
#     plt.savefig(fname, dpi=80)
#     plt.close()

In [15]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ── pre-compute global axis limits ──────────────────────────────────────────
all_x = np.concatenate([
    stream_model_half['part_xv'][:, :, 0].ravel(),
    stream_model_stat['part_xv'][:, :, 0].ravel(),
    stream_model_half['prog_xv'][:, 0].ravel(),
    stream_model_stat['prog_xv'][:, 0].ravel(),
])
all_y = np.concatenate([
    stream_model_half['part_xv'][:, :, 1].ravel(),
    stream_model_stat['part_xv'][:, :, 1].ravel(),
    stream_model_half['prog_xv'][:, 1].ravel(),
    stream_model_stat['prog_xv'][:, 1].ravel(),
])
all_z = np.concatenate([
    stream_model_half['part_xv'][:, :, 2].ravel(),
    stream_model_stat['part_xv'][:, :, 2].ravel(),
    stream_model_half['prog_xv'][:, 2].ravel(),
    stream_model_stat['prog_xv'][:, 2].ravel(),
])

def safe_lim(arr, pad=0.05):
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return (-1, 1)
    lo, hi = finite.min(), finite.max()
    span = hi - lo if hi != lo else 1.0
    return (lo - pad * span, hi + pad * span)

xlim = safe_lim(all_x)
ylim = safe_lim(all_y)
zlim = safe_lim(all_z)

# ── frame loop (reversed: starts at present day, plays backward) ─────────────
n_particles_h, n_timesteps_h, _ = stream_model_half['part_xv'].shape
n_particles_s, n_timesteps_s, _ = stream_model_stat['part_xv'].shape

frames_dir = 'mg_pres_half_mass_evolve_gif'
os.makedirs(frames_dir, exist_ok=True)

for frame_idx, t in enumerate(range(n_timesteps_h - 1, -1, -1)):

    # time label: 0 Gyr at present day, goes negative going back in time
    age = -((n_timesteps_h - 1 - t) * Age_stream_inGyr / (n_timesteps_h - 1))

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_facecolor('#F6F6F6')
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('none')
    ax.yaxis.pane.set_edgecolor('none')
    ax.zaxis.pane.set_edgecolor('none')
    ax.grid(False)
    ax.set_axis_off()

    # ── fixed axis limits ────────────────────────────────────────────────────
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)

    # ── stream particles ─────────────────────────────────────────────────────
    ax.scatter3D(
        stream_model_half['part_xv'][:, t, 0],
        stream_model_half['part_xv'][:, t, 1],
        stream_model_half['part_xv'][:, t, 2],
        s=2, c='tab:pink', alpha=0.6, label='Evolving',
    )
    ax.scatter3D(
        stream_model_stat['part_xv'][:, t, 0],
        stream_model_stat['part_xv'][:, t, 1],
        stream_model_stat['part_xv'][:, t, 2],
        s=2, c='#703BE7', alpha=0.6, label='Static',
    )

    # ── full orbit ghost trails ──────────────────────────────────────────────
    ax.plot3D(
        stream_model_half['prog_xv'][:, 0],
        stream_model_half['prog_xv'][:, 1],
        stream_model_half['prog_xv'][:, 2],
        c='tab:pink', linestyle=':', linewidth=1, alpha=0.25,
    )
    ax.plot3D(
        stream_model_stat['prog_xv'][:, 0],
        stream_model_stat['prog_xv'][:, 1],
        stream_model_stat['prog_xv'][:, 2],
        c='#703BE7', linestyle=':', linewidth=1, alpha=0.25,
    )

    # ── progenitor trails up to current timestep ─────────────────────────────
    ax.plot3D(
        stream_model_half['prog_xv'][:t + 1, 0],
        stream_model_half['prog_xv'][:t + 1, 1],
        stream_model_half['prog_xv'][:t + 1, 2],
        c='tab:pink', linestyle='--', linewidth=2, alpha=0.5,
    )
    ax.plot3D(
        stream_model_stat['prog_xv'][:t + 1, 0],
        stream_model_stat['prog_xv'][:t + 1, 1],
        stream_model_stat['prog_xv'][:t + 1, 2],
        c='#703BE7', linestyle='--', linewidth=2, alpha=0.5,
    )

    # ── progenitor current-position dots ─────────────────────────────────────
    ax.scatter3D(
        stream_model_half['prog_xv'][t, 0],
        stream_model_half['prog_xv'][t, 1],
        stream_model_half['prog_xv'][t, 2],
        s=20, c='tab:pink', zorder=5,
    )
    ax.scatter3D(
        stream_model_stat['prog_xv'][t, 0],
        stream_model_stat['prog_xv'][t, 1],
        stream_model_stat['prog_xv'][t, 2],
        s=20, c='#703BE7', zorder=5,
    )

    # ── legend ───────────────────────────────────────────────────────────────
    legend = ax.legend(loc='upper left', fontsize=10, frameon=False)
    for text in legend.get_texts():
        text.set_color('black')

    # ── time label ───────────────────────────────────────────────────────────
    ax.text2D(0.05, 0.05, f't = {age:.2f} Gyr', color='black', fontsize=12,
              transform=ax.transAxes)

    # ── rotating camera ───────────────────────────────────────────────────────
    elev = 20 + 10 * np.sin(2 * np.pi * frame_idx / n_timesteps_h)
    azim = (frame_idx / n_timesteps_h) * 360
    ax.view_init(elev=elev, azim=azim)

    fname = f'{frames_dir}/step_{frame_idx:04d}.png'
    plt.savefig(fname, dpi=80, bbox_inches='tight')
    plt.close()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from mpl_toolkits.mplot3d import Axes3D

# ── pre-compute global axis limits ──────────────────────────────────────────
all_x = np.concatenate([
    stream_model_half['part_xv'][:, :, 0].ravel(),
    stream_model_stat['part_xv'][:, :, 0].ravel(),
    stream_model_half['prog_xv'][:, 0].ravel(),
    stream_model_stat['prog_xv'][:, 0].ravel(),
])
all_y = np.concatenate([
    stream_model_half['part_xv'][:, :, 1].ravel(),
    stream_model_stat['part_xv'][:, :, 1].ravel(),
    stream_model_half['prog_xv'][:, 1].ravel(),
    stream_model_stat['prog_xv'][:, 1].ravel(),
])
all_z = np.concatenate([
    stream_model_half['part_xv'][:, :, 2].ravel(),
    stream_model_stat['part_xv'][:, :, 2].ravel(),
    stream_model_half['prog_xv'][:, 2].ravel(),
    stream_model_stat['prog_xv'][:, 2].ravel(),
])

def safe_lim(arr, pad=0.05):
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return (-1, 1)
    lo, hi = finite.min(), finite.max()
    span = hi - lo if hi != lo else 1.0
    return (lo - pad * span, hi + pad * span)

xlim = safe_lim(all_x)
ylim = safe_lim(all_y)
zlim = safe_lim(all_z)

# ── pre-compute 3D separation at every timestep ──────────────────────────────
n_particles_h, n_timesteps_h, _ = stream_model_half['part_xv'].shape
n_particles_s, n_timesteps_s, _ = stream_model_stat['part_xv'].shape

separation = np.sqrt(
    (stream_model_half['prog_xv'][:, 0] - stream_model_stat['prog_xv'][:, 0])**2 +
    (stream_model_half['prog_xv'][:, 1] - stream_model_stat['prog_xv'][:, 1])**2 +
    (stream_model_half['prog_xv'][:, 2] - stream_model_stat['prog_xv'][:, 2])**2
)

# time axis for inset: 0 at present (last timestep), negative going back
time_axis = np.linspace(-Age_stream_inGyr, 0, n_timesteps_h)

# ── disk ellipse geometry (fixed in 3D) ─────────────────────────────────────
# MW disk: flat ellipse in the X-Y plane at Z=0, radius ~15 kpc
disk_theta = np.linspace(0, 2 * np.pi, 200)
disk_r = 15.0   # kpc
disk_x = disk_r * np.cos(disk_theta)
disk_y = disk_r * np.sin(disk_theta)
disk_z = np.zeros_like(disk_theta)

# ── fixed camera angle ───────────────────────────────────────────────────────
# choose elev/azim that shows both orbital planes well
ELEV = 25
AZIM = 50

frames_dir = 'cray_fig_mary_gates_gif'
os.makedirs(frames_dir, exist_ok=True)

for frame_idx, t in enumerate(range(n_timesteps_h - 1, -1, -1)):

    age = -((n_timesteps_h - 1 - t) * Age_stream_inGyr / (n_timesteps_h - 1))

    # ── figure: 3D main + inset ──────────────────────────────────────────────
    fig = plt.figure(figsize=(10, 8))
    fig.patch.set_facecolor('#F6F6F6')

    ax = fig.add_subplot(111, projection='3d')
    ax.set_facecolor('#F6F6F6')
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('none')
    ax.yaxis.pane.set_edgecolor('none')
    ax.zaxis.pane.set_edgecolor('none')
    ax.grid(False)
    ax.set_axis_off()

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)

    # ── MW disk reference ────────────────────────────────────────────────────
    ax.plot3D(disk_x, disk_y, disk_z,
              c='gray', linewidth=0.8, alpha=0.3, linestyle='-')
    # filled disk shading: a few concentric rings for depth
    for r in np.linspace(2, disk_r, 6):
        ax.plot3D(r * np.cos(disk_theta), r * np.sin(disk_theta), disk_z,
                  c='gray', linewidth=0.3, alpha=0.12)
    # galactic center marker
    ax.scatter3D(0, 0, 0, s=40, c='gold', marker='*', zorder=10,
                 edgecolors='goldenrod', linewidths=0.5)

    # ── stream particles ─────────────────────────────────────────────────────
    ax.scatter3D(
        stream_model_half['part_xv'][:, t, 0],
        stream_model_half['part_xv'][:, t, 1],
        stream_model_half['part_xv'][:, t, 2],
        s=2, c='tab:pink', alpha=0.5, label='Evolving',
    )
    ax.scatter3D(
        stream_model_stat['part_xv'][:, t, 0],
        stream_model_stat['part_xv'][:, t, 1],
        stream_model_stat['part_xv'][:, t, 2],
        s=2, c='#703BE7', alpha=0.5, label='Static',
    )

    # ── full ghost orbit trails ──────────────────────────────────────────────
    ax.plot3D(
        stream_model_half['prog_xv'][:, 0],
        stream_model_half['prog_xv'][:, 1],
        stream_model_half['prog_xv'][:, 2],
        c='tab:pink', linestyle=':', linewidth=1, alpha=0.2,
    )
    ax.plot3D(
        stream_model_stat['prog_xv'][:, 0],
        stream_model_stat['prog_xv'][:, 1],
        stream_model_stat['prog_xv'][:, 2],
        c='#703BE7', linestyle=':', linewidth=1, alpha=0.2,
    )

    # ── progenitor trails up to current timestep ─────────────────────────────
    ax.plot3D(
        stream_model_half['prog_xv'][:t + 1, 0],
        stream_model_half['prog_xv'][:t + 1, 1],
        stream_model_half['prog_xv'][:t + 1, 2],
        c='tab:pink', linestyle='--', linewidth=2, alpha=0.7,
    )
    ax.plot3D(
        stream_model_stat['prog_xv'][:t + 1, 0],
        stream_model_stat['prog_xv'][:t + 1, 1],
        stream_model_stat['prog_xv'][:t + 1, 2],
        c='#703BE7', linestyle='--', linewidth=2, alpha=0.7,
    )

    # ── progenitor current-position dots ─────────────────────────────────────
    ax.scatter3D(
        stream_model_half['prog_xv'][t, 0],
        stream_model_half['prog_xv'][t, 1],
        stream_model_half['prog_xv'][t, 2],
        s=40, c='tab:pink', zorder=5, edgecolors='white', linewidths=0.5,
    )
    ax.scatter3D(
        stream_model_stat['prog_xv'][t, 0],
        stream_model_stat['prog_xv'][t, 1],
        stream_model_stat['prog_xv'][t, 2],
        s=40, c='#703BE7', zorder=5, edgecolors='white', linewidths=0.5,
    )

    # ── line connecting the two progenitors at this timestep ────────────────
    ax.plot3D(
        [stream_model_half['prog_xv'][t, 0], stream_model_stat['prog_xv'][t, 0]],
        [stream_model_half['prog_xv'][t, 1], stream_model_stat['prog_xv'][t, 1]],
        [stream_model_half['prog_xv'][t, 2], stream_model_stat['prog_xv'][t, 2]],
        c='white', linewidth=1.2, alpha=0.6, linestyle='-',
    )

    # ── legend ───────────────────────────────────────────────────────────────
    legend = ax.legend(loc='upper left', fontsize=10, frameon=False)
    for text in legend.get_texts():
        text.set_color('black')

    # ── time label ───────────────────────────────────────────────────────────
    ax.text2D(0.05, 0.05, f't = {age:.2f} Gyr', color='black', fontsize=11,
              transform=ax.transAxes)

    # ── fixed camera ─────────────────────────────────────────────────────────
    ax.view_init(elev=ELEV, azim=AZIM)

    # ── inset: 3D separation vs time ─────────────────────────────────────────
    # placed in lower-right of the figure in figure coordinates
    ax_inset = fig.add_axes([0.62, 0.08, 0.30, 0.22])
    ax_inset.set_facecolor('#EBEBEB')

    # full separation curve (faint)
    ax_inset.plot(time_axis, separation, color='gray', linewidth=1.0, alpha=0.4)

    # highlight the portion revealed so far (from present back to current age)
    revealed_mask = time_axis >= age
    ax_inset.plot(time_axis[revealed_mask], separation[revealed_mask],
                  color='white', linewidth=1.5, alpha=0.9)

    # current moment dot
    ax_inset.scatter(age, separation[t], s=25, color='white', zorder=5)

    ax_inset.set_xlabel('Time (Gyr)', fontsize=7, color='black')
    ax_inset.set_ylabel('|Δr| (kpc)', fontsize=7, color='black')
    ax_inset.set_title('Progenitor separation', fontsize=7, color='black', pad=3)
    ax_inset.tick_params(labelsize=6, colors='black')
    for spine in ax_inset.spines.values():
        spine.set_edgecolor('gray')
    ax_inset.set_xlim(time_axis[0], 0)
    ax_inset.set_ylim(0, separation.max() * 1.1)

    fname = f'{frames_dir}/step_{frame_idx:04d}.png'
    plt.savefig(fname, dpi=80, bbox_inches='tight')
    plt.close()